# Introduction

This mini-project uses a synthetic credit dataset to demonstrate a model evaluation workflow. It separates model fitting, threshold selection, and final assessment: training data fit the model, validation data guide early stopping and threshold choice, and the test set is reserved for one final evaluation. The data are illustrative and do not represent real borrowers or a production credit policy.


# Import packages

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# Generate a synthetic dataset

Each feature is generated within a plausible range for this demonstration. The target is sampled from a specified probability, so this is not a real credit-risk dataset and its learned relationships should not be used for lending decisions.


In [ ]:
rng = np.random.default_rng(SEED)
n = 5000

income = np.clip(rng.normal(60000, 18000, n), 10000, 200000)
debt_to_income = rng.beta(2, 5, n)
credit_utilization = rng.beta(2, 3, n)
late_payments = rng.poisson(0.4, n)
employment_years = np.clip(rng.exponential(5, n), 0, 45)
loan_amount = np.clip(rng.normal(15000, 5000, n), 1000, 100000)
age = np.clip(rng.normal(40, 12, n), 18, 90)

# Center and scale the risk score so the synthetic default rate is not extreme.
risk_score = (
    2.5 * (debt_to_income - 0.3)
    + 2.0 * (credit_utilization - 0.4)
    + 0.7 * late_payments
    - 0.000015 * (income - 60000)
    - 0.05 * (employment_years - 5)
    + 0.00003 * (loan_amount - 15000)
    - 1.5
)
prob_default = 1 / (1 + np.exp(-risk_score))
default = (rng.random(n) < prob_default).astype(int)

df = pd.DataFrame({
    "income": income,
    "debt_to_income": debt_to_income,
    "credit_utilization": credit_utilization,
    "late_payments": late_payments,
    "employment_years": employment_years,
    "loan_amount": loan_amount,
    "age": age,
    "default": default,
})
print(df.describe().round(2))
print("Synthetic default rate:", round(df["default"].mean(), 3))


# Feature engineering and data splits

Split the data before fitting preprocessing steps. The training split fits the scaler and model; the validation split supports early stopping and threshold selection. The test split is held back for the final evaluation.


In [ ]:
features = [
    "income", "debt_to_income", "credit_utilization", "late_payments",
    "employment_years", "loan_amount", "age",
]
X = df[features].to_numpy(dtype=np.float32)
y = df["default"].to_numpy(dtype=np.int64)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

X_train_tensor = torch.tensor(X_train_scaled)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_tensor = torch.tensor(X_val_scaled)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled)


## Baseline model

A logistic regression provides a simple reference point. Fit it on training data and use validation data for a comparable initial assessment. The test set remains reserved for the final assessment after modeling decisions are complete.


In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train_scaled, y_train)
baseline_val_prob = baseline.predict_proba(X_val_scaled)[:, 1]
print("Logistic regression validation ROC AUC:", roc_auc_score(y_val, baseline_val_prob))
print("Logistic regression validation average precision:", average_precision_score(y_val, baseline_val_prob))


# Model

The network returns logits (unbounded scores). `BCEWithLogitsLoss` combines the sigmoid operation and binary cross-entropy in a numerically stable loss. Apply sigmoid only when probabilities are needed.


In [ ]:
class CreditRiskModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.network(x)


# Training with validation-based early stopping

The validation loss selects the training epoch. It is not used to report final performance. Restoring the best validation checkpoint prevents later epochs from replacing the best observed model.


In [ ]:
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

model = CreditRiskModel(input_dim=len(features))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
best_val_loss = float("inf")
best_state = None
early_stopping_patience = 5
stale_epochs = 0

for epoch in range(50):
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_tensor), y_val_tensor).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}: train loss={train_loss / len(train_loader):.4f}, validation loss={val_loss:.4f}")
    if stale_epochs >= early_stopping_patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

model.load_state_dict(best_state)


# Select a threshold using validation data, then assess once on the test set

A threshold converts a predicted probability to a class label. The best threshold depends on the costs and constraints of a real decision process. Here, F1 is used only as an illustrative selection rule; it is not a lending policy.


In [ ]:
model.eval()
with torch.no_grad():
    val_prob = torch.sigmoid(model(X_val_tensor)).numpy().ravel()
    y_prob = torch.sigmoid(model(X_test_tensor)).numpy().ravel()

thresholds = np.arange(0.1, 0.91, 0.01)
val_f1 = [f1_score(y_val, (val_prob >= t).astype(int), zero_division=0) for t in thresholds]
selected_threshold = float(thresholds[int(np.argmax(val_f1))])
print(f"Validation-selected threshold (F1 rule): {selected_threshold:.2f}")

y_pred = (y_prob >= selected_threshold).astype(int)
print("Test confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("Test classification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Test ROC AUC:", roc_auc_score(y_test, y_prob))
print("Test average precision:", average_precision_score(y_test, y_prob))
print("Test precision:", precision_score(y_test, y_pred, zero_division=0))
print("Test recall:", recall_score(y_test, y_pred, zero_division=0))
print("Test F1:", f1_score(y_test, y_pred, zero_division=0))


## ROC curve

The ROC curve summarizes ranking performance across thresholds. The plotted curve uses the held-out test probabilities for final assessment.


In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc_score(y_test, y_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Precision-recall curve

The precision–recall curve shows the tradeoff across thresholds. Average precision must be calculated from continuous scores such as predicted probabilities, not hard class labels. Compare average precision with the positive-class prevalence shown above.


In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"AP = {average_precision_score(y_test, y_prob):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

## Threshold analysis

Inspect candidate thresholds on validation data only. This table helps show the tradeoff behind the illustrative F1 selection rule; do not use the test set to choose a threshold.


In [ ]:
threshold_rows = []
for threshold in np.arange(0.1, 0.91, 0.1):
    preds = (val_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, preds, zero_division=0),
        "recall": recall_score(y_val, preds, zero_division=0),
        "f1": f1_score(y_val, preds, zero_division=0),
        "predicted_default_rate": preds.mean(),
    })
threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.round(3))


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.title("Threshold Tradeoff")
plt.legend()
plt.show()

In [ ]:
results = pd.DataFrame(scaler.inverse_transform(X_test_scaled), columns=features)
results["actual"] = y_test
results["probability"] = y_prob
results["prediction"] = y_pred
results["error_type"] = "correct"
results.loc[(results["actual"] == 0) & (results["prediction"] == 1), "error_type"] = "false_positive"
results.loc[(results["actual"] == 1) & (results["prediction"] == 0), "error_type"] = "false_negative"
print(results["error_type"].value_counts())


False positives are applicants predicted as likely to default who did not default. False negatives are applicants predicted as safe who did default. In credit scoring, both matter, but they represent different business risks.

In [ ]:
error_summary = results.groupby("error_type")[features + ["probability"]].mean()
print(error_summary)

A compact plot can reveal whether errors cluster in certain feature ranges.

In [ ]:
plt.figure(figsize=(7, 5))

for error_type in ["correct", "false_positive", "false_negative"]:
    subset = results[results["error_type"] == error_type]
    plt.scatter(
        subset["debt_to_income"],
        subset["credit_utilization"],
        label=error_type,
        alpha=0.5
    )

plt.xlabel("Debt-to-income ratio")
plt.ylabel("Credit utilization")
plt.title("Error Analysis by Credit Risk Features")
plt.legend()
plt.show()

# Explainability

## SHAP values

For a credit-scoring model, explanations can describe how model inputs relate to its predictions. Explanations describe model behavior; they do not establish causal effects or justify a decision by themselves.


In [ ]:
import shap

background = X_train_tensor[:200]
sample = X_test_tensor[:200]

explainer = shap.DeepExplainer(model, background)
shap_values = explainer.shap_values(sample)

shap_values_array = shap_values[0] if isinstance(shap_values, list) else shap_values

For a single applicant, SHAP values help identify which features pushed the prediction toward higher or lower risk. High credit utilization, high debt-to-income ratio, and repeated late payments may increase predicted risk. Higher income or longer employment history may reduce it.

In [ ]:
shap.summary_plot(
    shap_values_array,
    sample.numpy(),
    feature_names=features
)

In [ ]:
shap.summary_plot(
    shap_values_array,
    sample.numpy(),
    feature_names=features,
    plot_type="bar"
)

For an individual applicant, SHAP can show which features pushed the predicted default probability upward or downward.

## LIME

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

def predict_proba_for_lime(data):
    data_tensor = torch.tensor(np.asarray(data), dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        default_prob = torch.sigmoid(model(data_tensor)).numpy().ravel()
    return np.column_stack([1 - default_prob, default_prob])


In [ ]:
applicant_index = 0

lime_explainer = LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=features,
    class_names=["non_default", "default"],
    mode="classification",
    random_state=SEED,
)

lime_exp = lime_explainer.explain_instance(
    data_row=X_test_scaled[applicant_index],
    predict_fn=predict_proba_for_lime,
    num_features=6,
)
lime_exp.show_in_notebook(show_table=True)


In [ ]:
fig = lime_exp.as_pyplot_figure()
plt.title("LIME Explanation for One Applicant")
plt.show()

SHAP and LIME should not be interpreted as causal truth. They describe how a fitted model responds to inputs under the explainer's assumptions. For example, if SHAP attributes a higher predicted risk to high credit utilization, that does not prove that changing utilization alone would change a borrower's true default risk.


## Combine explanation with error analysis

In [ ]:
false_negatives = results[results["error_type"] == "false_negative"]
false_positives = results[results["error_type"] == "false_positive"]

print("False negatives with highest predicted safety:")
print(false_negatives.sort_values("probability").head())

print("False positives with highest predicted risk:")
print(false_positives.sort_values("probability", ascending=False).head())

In [ ]:
applicant_index = 0

print("Predicted default probability:", y_prob[applicant_index])

applicant_original = pd.DataFrame(
    scaler.inverse_transform(X_test_scaled[applicant_index].reshape(1, -1)),
    columns=features
)

print(applicant_original)

In [ ]:
confident_errors = results[
    ((results["error_type"] == "false_positive") & (results["probability"] > 0.8)) |
    ((results["error_type"] == "false_negative") & (results["probability"] < 0.2))
]

print(confident_errors)

In [ ]:
if len(confident_errors) > 0:
    idx = confident_errors.index[0]
    lime_exp = lime_explainer.explain_instance(
        data_row=X_test_scaled[idx],
        predict_fn=predict_proba_for_lime,
        num_features=6,
    )
    fig = lime_exp.as_pyplot_figure()
    plt.title("LIME Explanation for a Confident Error")
    plt.show()
else:
    print("No confident errors at the illustrative cutoffs.")
